# 5-HT2A Membrane Build — Stage 1: packmol-memgen packing

Builds the DPPC bilayer + water + ions around your cleaned 5-HT2A receptor using **packmol-memgen** (AmberTools) as a geometry packer only. Stage 2 then applies **CHARMM36** parameters to keep the force field consistent with your D2 system.

**Why two stages:** packmol-memgen places the lipids/water/ions (geometry); CHARMM36 supplies the parameters (physics). Separating them lets us keep D2 consistency and debug each step independently. The tricky part — converting packmol-memgen's Amber lipid naming to CHARMM naming — is isolated in Stage 2.

**Input:** `5ht2a_clean.pdb` (your cleaned receptor: chain A, BRIL removed, residues 69-265 + 313-399).
**Output:** `bilayer_5ht2a.pdb` — receptor packed in a DPPC bilayer with water and 0.15 M NaCl.

**Run on the pod** (needs AmberTools; CPU is fine for packing — no GPU yet).

---
## Part 1 — Install AmberTools (one-time, in the mddock env)

In [ ]:
# Cell 1: Check for / install AmberTools (provides packmol-memgen)
import shutil, subprocess, sys
have = shutil.which("packmol-memgen")
print("packmol-memgen found:", have)
if not have:
    print("Installing AmberTools via conda (a few minutes)...")
    # Run this in a terminal if the notebook conda call is awkward:
    #   conda install -n mddock -c conda-forge ambertools -y
    rc = subprocess.run("conda install -c conda-forge ambertools -y",
                        shell=True, capture_output=True, text=True)
    print(rc.stdout[-2000:]); print(rc.stderr[-1000:])
    print("packmol-memgen now:", shutil.which("packmol-memgen"))
print("\nIf install is slow/awkward here, run in a pod TERMINAL instead:")
print("   conda install -n mddock -c conda-forge ambertools -y")

---
## Part 2 — Orient the receptor in the membrane

packmol-memgen needs the protein pre-oriented with the membrane normal along z. The cleanest source is the **OPM database** (Orientations of Proteins in Membranes). Two options below.

In [ ]:
# Cell 2: Get a membrane-oriented receptor
# OPTION A (preferred): download the OPM-oriented 6A93, then re-strip to your clean selection.
# OPTION B: let packmol-memgen orient via memembed (drop --preoriented in Cell 3).
import os, urllib.request
from pathlib import Path
WORKDIR = Path("/workspace/5ht2a_system"); os.chdir(WORKDIR)

CLEAN = WORKDIR/"5ht2a_clean.pdb"
assert CLEAN.exists(), "Upload 5ht2a_clean.pdb to the pod first"

# Try OPM oriented structure for 6A93
opm_path = WORKDIR/"6a93_opm.pdb"
try:
    urllib.request.urlretrieve("https://opm-assets.storage.googleapis.com/pdb/6a93.pdb", str(opm_path))
    print(f"✓ Downloaded OPM-oriented 6A93 ({opm_path.stat().st_size/1e6:.2f} MB)")
    print("  -> re-run your Cell-5 strip on THIS file to get an oriented clean receptor,")
    print("     OR just use 5ht2a_clean.pdb with memembed orientation (Option B).")
    USE_PREORIENTED = True
except Exception as e:
    print("OPM download failed:", e)
    print("-> Will use Option B: packmol-memgen orients via memembed (no --preoriented).")
    USE_PREORIENTED = False

**Decision:** If the OPM file downloaded, the most consistent route is to re-run your stripping (chain A, drop BRIL 1001-1106, drop hetero) on `6a93_opm.pdb` to get an *oriented* clean receptor, and use `--preoriented` in Cell 3. If OPM failed, skip orientation and let packmol-memgen's memembed orient it (omit `--preoriented`). Tell me which path and I'll confirm the exact Cell 3 flags.

---
## Part 3 — Pack the DPPC bilayer

In [ ]:
# Cell 3: Run packmol-memgen (geometry only; DPPC bilayer + water + 0.15 M NaCl)
# Flags chosen to match the D2 system (DPPC) and to NOT alter your clean receptor.
import subprocess
INPUT_PDB = "5ht2a_clean.pdb"        # or the oriented version from Cell 2
PREORIENT = "--preoriented" if False else ""   # set True if using an OPM-oriented input

cmd = (
    f"packmol-memgen --pdb {INPUT_PDB} "
    f"--lipids DPPC "                      # match D2's DPPC bilayer
    f"--salt --salt_c Na+ --saltcon 0.15 " # physiological NaCl
    f"--dist 12 --dist_wat 17.5 "          # protein-box and water padding (Å)
    f"{PREORIENT} "
    f"--notprotonate --nottrim "           # do NOT modify the already-clean receptor
    f"--log packmol_memgen.log"
)
print("Running:\n", cmd, "\n")
rc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(rc.stdout[-3000:])
if rc.returncode != 0:
    print("STDERR:", rc.stderr[-2000:])
    print("\n⚠ packmol-memgen failed — send Claude the stderr above.")
else:
    print("\n✓ packing finished")

In [ ]:
# Cell 4: Locate and sanity-check the packed output
import glob
out = sorted(glob.glob("bilayer_*.pdb"))
print("Packed files:", out)
assert out, "No bilayer_*.pdb produced — check the packmol-memgen log"
PACKED = out[0]

# Quick composition check
from collections import Counter
resn = Counter()
with open(PACKED) as f:
    for line in f:
        if line.startswith(("ATOM","HETATM")):
            resn[line[17:20].strip()] += 1
print(f"\n{PACKED} residue/atom composition (top):")
for k,c in resn.most_common(12):
    print(f"  {k}: {c} atoms")
print("\nExpect: protein residues + DPPC (or PA/PC split names) + WAT/water + Na+/Cl-")
print("Note the lipid naming — Stage 2 converts it to CHARMM convention.")

---
## Part 4 — Hand-off to Stage 2

Stage 1 produced `bilayer_5ht2a.pdb` (receptor + DPPC + water + ions, packed geometry).

**Before Stage 2, send Claude the Cell 4 composition output** — specifically the lipid and water/ion residue names packmol-memgen used (e.g. `DPPC` vs split `PA`/`PC`/`OL`, `WAT` vs `HOH`, `Na+`/`Cl-` vs `SOD`/`CLA`). Those exact names determine the CHARMM naming-conversion map in Stage 2. Getting them right is the one fiddly step, so we confirm them from real output rather than guessing.

**What Stage 2 will do:** convert the lipid/water/ion naming to CHARMM36 convention, apply CHARMM36 parameters, build the OpenMM system, and equilibrate — producing `ht2a_gmx_top.pkl` + `ht2a_final_equilibrated.xml`, the inputs Notebook 2 needs.